# Dynamic Programming

**Reinforcement Learning Course** — Notebook 03

We start from a fixed policy and compute its value function, then find the optimal policy $\pi^*$ using Policy Iteration and Value Iteration — all on the Grid World from Notebook 02.

The reward is sparse: **★ = +1**, everywhere else = 0.

In [ ]:
!pip install numpy plotly

In [ ]:
import sys; sys.path.insert(0, "..")
from practical_case import markov_env as env

## Part 1 — Value function of a fixed policy

$V^\pi(s)$ measures the expected discounted reward starting from $s$ and following $\pi$ forever:

$$V^\pi(s) = \mathbb{E}\!\left[\sum_{t=0}^{\infty} \gamma^t\, r(S_t) \;\middle|\; S_0 = s,\, \pi\right]$$

We evaluate three policies at $\gamma = 0.95$. **Bright cells = high expected return.**  
The arrows show the mean drift of each policy — observe how the value landscape reflects where the policy leads the agent.

In [ ]:
r     = env.default_reward()
gamma = 0.95

policies = {
    "Spiral CCW": env.spiral_ccw(),
    "Spiral CW":  env.spiral_cw(),
    "Rightward":  env.rightward(),
}

for name, pi in policies.items():
    V = env.policy_eval(pi, r, gamma)
    env.show_value(V, pi=pi, title=f"Vᵖ — {name}").show()

## Part 2 — Effect of the discount factor $\gamma$

$\gamma$ controls the planning horizon:

- **Small $\gamma$** → myopic agent: only reward reachable in a few steps counts.
- **$\gamma \to 1$** → far-sighted agent: value propagates across the entire grid.

Same policy (spiral CCW), three discount values. Notice how the value front expands as $\gamma$ grows.

In [ ]:
pi_ccw = env.spiral_ccw()

for g in [0.5, 0.9, 0.99]:
    V = env.policy_eval(pi_ccw, r, gamma=g)
    env.show_value(V, title=f"γ = {g}").show()

## Part 3 — Policy Iteration

Starting from a **uniform random policy**, PI alternates between:

1. **Evaluate** — compute $V^{\pi_k}$ exactly via Bellman backups.
2. **Improve** — act greedily: $\pi_{k+1}(s) = \arg\max_a \bigl[r(s) + \gamma \sum_{s'} P(s'|s,a)\,V^{\pi_k}(s')\bigr]$.

Each figure shows one PI step — watch the arrows progressively orient toward ★.

In [ ]:
pi_hist = env.policy_iteration(r, gamma=0.95)

for k, (pi, V) in enumerate(pi_hist):
    env.show_value(V, pi=pi, title=f"PI step {k}").show()

## Part 4 — Value Iteration & convergence

VI applies the Bellman optimality operator directly, without a full inner evaluation loop:

$$V_{k+1}(s) = \max_a \Bigl[r(s) + \gamma \sum_{s'} P(s'|s,a)\,V_k(s')\Bigr]$$

The first plot shows $V^*$ with the optimal policy. The convergence plot shows $\|V_k - V^*\|_\infty$ on a log scale for both algorithms.

> **Note:** one PI step hides an inner evaluation loop — the x-axes are not directly comparable in compute cost. PI converges in fewer *outer* steps; VI converges in fewer *Bellman backups* per step.

In [ ]:
vi_hist = env.value_iteration(r, gamma=0.95)
V_star  = vi_hist[-1]
pi_star = env.greedy_policy(V_star, r, gamma=0.95)

env.show_value(V_star, pi=pi_star, title="V* — optimal value function").show()
env.vi_pi_convergence(vi_hist, pi_hist, V_star).show()

## Part 5 — (Optional) Design your own reward

Define `my_reward` below — any function $r : \mathcal{S} \to \mathbb{R}$.  
Policy Iteration will find the optimal policy for your objective.

**Hint:** try multiple goals, a negative penalty zone, or a reward that depends on the row/column.

In [ ]:
import numpy as np

def my_reward() -> np.ndarray:
    r = np.zeros(env.N)
    # ── your reward here ──────────────────────────────────────────────

    # ─────────────────────────────────────────────────────────────────
    return r

# Uncomment once implemented:
# r_custom  = my_reward()
# pi_hist_c = env.policy_iteration(r_custom, gamma=0.95)
# V_star_c  = pi_hist_c[-1][1]
# env.show_value(V_star_c, pi=pi_hist_c[-1][0], title="V* — custom reward").show()